# ChargeGrid AI — Sprint 04: pipeline de avaliação no Colab

Roda o golden dataset nas duas versões do chatbot (Sprint 2 e Sprint 3), julga as respostas com o LLM juiz e gera as tabelas.
É o mesmo pipeline de `evals/sprint4/` (ver `evals/sprint4/README.md`), só que executado no Colab.

**Antes de rodar**, cadastre nos **Secrets do Colab** (ícone de chave na barra lateral) e libere o acesso deste notebook:

| Secret | Uso |
|---|---|
| `GROQ_API_KEY` | versão Sprint 2 (`openai/gpt-oss-20b` na Groq) |
| `GOOGLE_API_KEY` | versão Sprint 3 (`gemini-3.6-flash`) |
| `OPENAI_API_KEY` | juiz (`gpt-4o-mini`) |

Nenhuma chave é escrita no código nem impressa.

## 1. Código do projeto

In [ ]:
import os
import zipfile
from pathlib import Path

ZIP = "chargedgrid-ai-sprint04.zip"   # gerado com: git archive --prefix=chargedgrid-ai/ sprint-04

if not Path("chargedgrid-ai").exists():
    if not Path(ZIP).exists():
        from google.colab import files
        print(f"Envie o arquivo {ZIP}")
        files.upload()
    zipfile.ZipFile(ZIP).extractall(".")

os.chdir("chargedgrid-ai")
print("Pasta de trabalho:", os.getcwd())

## 2. Dependências

Mesmos pins da Sprint 2 e da Sprint 3 (`requirements.txt`). Leva alguns minutos.

In [ ]:
!pip install -q -r requirements.txt

## 3. Chaves (Secrets do Colab)

In [ ]:
from google.colab import userdata

for nome in ("GROQ_API_KEY", "GOOGLE_API_KEY", "OPENAI_API_KEY"):
    valor = userdata.get(nome)
    if not valor:
        raise RuntimeError(f"Cadastre {nome} nos Secrets do Colab e libere o acesso deste notebook.")
    os.environ[nome] = valor

# Mesma configuracao do .env.example: raciocinio do Gemini desligado (ver agent_core/config.py).
os.environ["MODEL_THINKING"] = "0"
os.environ["PYTHONIOENCODING"] = "utf-8"
print("Chaves carregadas.")

## 4. Índice do RAG e testes sem chave

O índice FAISS é gerado a partir dos 12 documentos de `app/rag/docs/` (59 chunks).

In [ ]:
!python create_vector_store.py
!python evals/smoke_offline.py
!python evals/sprint4/teste_offline.py

## 5. Respostas das duas versões

A Groq tem cota de 200.000 tokens por dia por modelo. Se a versão Sprint 2 parar por cota, rode de novo no dia seguinte com `--retomar`: ela continua dos casos que faltaram, sem repetir os que já têm resposta.

In [ ]:
!python evals/sprint4/gerar_respostas.py --versao sprint3

In [ ]:
!python evals/sprint4/gerar_respostas.py --versao sprint2
# !python evals/sprint4/gerar_respostas.py --versao sprint2 --retomar

## 6. Juiz

Julga as duas versões e também as respostas que a Sprint 03 avaliou à mão (para a comparação com a avaliação manual).

In [ ]:
!python evals/sprint4/juiz.py --versao sprint2
!python evals/sprint4/juiz.py --versao sprint3
!python evals/sprint4/juiz.py --sprint3-manual

## 7. Métricas e tabelas

In [ ]:
!python evals/sprint4/metricas.py

from IPython.display import Markdown, display
display(Markdown(Path("evals/sprint4/results/resultados.md").read_text(encoding="utf-8")))

## 8. Baixar os resultados

O zip traz respostas, julgamentos, `resumo.json` e `resultados.md`, que voltam para `evals/sprint4/results/` no repositório.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/resultados_sprint4", "zip", "evals/sprint4/results")
files.download("/content/resultados_sprint4.zip")